In [9]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [10]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        # Extended thinking is a single nested object.
        # Constraints: temperature must be 1, and max_tokens must exceed budget_tokens.
        params["thinking"] = {"type": "enabled", "budget_tokens": thinking_budget}
        params["temperature"] = 1.0
        params["max_tokens"] = thinking_budget + 4000

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [11]:
messages = []

add_user_message(messages, "Write a one paragraph summary of the following text"
                 )

chat(messages,thinking=True)

Message(id='msg_011Ccqzf4RxxRyfN4xpr6hq7', container=None, content=[ThinkingBlock(signature='ErkDCpQBCA8YAipAi2mUM61J1mbDevirskSgnascd9S7STMDkxPwAEbbCtJP3gU+PCNfBdVbSzSAlR4Sqi4EYRi2iRpd8rio0B7mDDIaY2xhdWRlLXNvbm5ldC00LTUtMjAyNTA5Mjk4AEIIdGhpbmtpbmdaJGI0ZjVlNmVjLTlhM2QtNGYwOC04MmMyLTczYWUyOWUzYThhOBIMIMGVP2XwHDJ5oGhhGgwF8DlYFOLPxNYR9w4iMGs9e+nzUaIUXP4erp1Ewe+EhPALpirnybJ8p/fV3Ea0fePSTBVI+5cQpkNZtXjNDCrRAULjksJ3mpmyp9DAuqhYzvNWoxTk8ApI5epGfXY6YCFhOoj0vKkg0dI04m8KgJ3ZVK7lgt9h/UBddoEEZVaHtkj5HW6Zt7QNd/Wrons25IDf4zbZBZZ1Y1UPd2XKjtON0o3j1/D2ilVyhC53BmsRNvqHH9LLdRA9rYqvfxe7+xi/I6v6jcTgAUVWoHqzRF1BPcbkdjqF+q7QNBXN62i+iz+4rJek48+wMf8UNShfzJqefaKgAKaPSjL2P0lMHrIDE4DolufKfcE/+L5rE1oFGSg+GAE=', thinking="The user wants me to write a one paragraph summary, but they haven't provided the text they want me to summarize. I should let them know that I need the text in order to complete their request.", type='thinking'), TextBlock(citations=None, text="I'd be happy to help you summarize a text, but I don